# Chavruta.AI — Stage 2: הרחבת האינדקס הקיים (Lightning AI · GPU)

**מקבל בקלט:** שני מאגרי HF החדשים מ-`fetch_wikisource_kaggle.ipynb`
(`chavruta-wikisource-kook`, `chavruta-wikisource-halacha`) **+ האינדקס המפורסם הקיים**
(`Yehuda-Rubin/chavruta-commercial-index`, 2,403,599 נקודות).

**עושה:**
1. משחזר את האינדקס הקיים ל-Qdrant מקומי (משתמש ב-`scripts/restore_commercial_snapshot.py`
   הקיים והמוכח — **לא** נכתב מחדש כאן).
2. מוריד את שני ה-JSONL החדשים ומטמיע **רק אותם** (bge-m3, GPU) — לא מטמיע מחדש את 2.4M
   הנקודות הקיימות. אותו קוד הטמעה בדיוק כמו `scripts/index_commercial_job.py::step_embed`.
3. `upsert` (לא `drop`+`recreate`!) את הנקודות החדשות **על גבי** הקולקציה המשוחזרת. `QdrantStore.upsert`
   הוא idempotent לפי `chunk_id` ([qdrant_store.py](../src/chavruta/store/qdrant_store.py)) —
   לא נוגע ב-2.4M הנקודות הקיימות.
4. בונה מחדש את ה-payload indexes (פעולה בטוחה לחזור עליה).
5. מפרסם snapshot חדש **למאגר HF חדש נפרד** — **לא דורס** את `chavruta-commercial-index` הקיים.
   שינוי הקולקציה בפרודקשן נשאר החלטה ידנית שלכם (להחליף repo ב-`restore_commercial_snapshot.py`
   כשתאשרו איכות).

⚠️ **לא נבדק על סביבת GPU אמיתית** (נכתב מסשן מחקר בלי גישה ל-Lightning/Qdrant). שלבים 2 ו-2.5
(הטמעה + upsert) הם קוד קיים שכבר רץ בפרודקשן — הסיכון האמיתי הוא בשלב 1 (שחזור) ו-4 (פרסום),
שהם חדשים כאן. הריצו על נתח קטן קודם (יש דגל `SMOKE_TEST` בקונפיג) לפני ריצה מלאה.


### 0. שכפול הריפו — כדי לייבא את `chavruta` + להריץ את `restore_commercial_snapshot.py` הקיים בלי לשכפל את הלוגיקה

In [ ]:
REPO_URL = "https://github.com/Yehuda-Rubin/Chavruta.AI.git"   # עדכנו לכתובת הריפו האמיתית שלכם
!git clone --depth 1 "{REPO_URL}" chavruta_repo 2>&1 | tail -5
import sys
sys.path.insert(0, "chavruta_repo/src")


### 1. בדיקת GPU + התקנות (גרסאות נעוצות — כמו ב-index_commercial_job.py)

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q "FlagEmbedding==1.3.4" "transformers==4.44.2" "huggingface_hub>=0.23" \
    "qdrant-client>=1.9" numpy requests


### 2. הגדרות

טוקן HF מתיבת קלט (כמו ב-Kaggle notebook) — לא Secrets, לא משתנה סביבה.

In [ ]:
import getpass

NAMESPACE = "Yehuda-Rubin"

# שני המאגרים החדשים מהמחברת הקודמת
NEW_TIER_REPOS = [
    f"{NAMESPACE}/chavruta-wikisource-kook",
    f"{NAMESPACE}/chavruta-wikisource-halacha",
]

EXISTING_INDEX_REPO = f"{NAMESPACE}/chavruta-commercial-index"     # קיים בפרודקשן — לקריאה בלבד
NEW_INDEX_REPO       = f"{NAMESPACE}/chavruta-extended-index-v1"    # יעד חדש — לא דורס את הקיים

COLLECTION = "chavruta_commercial"       # אותו שם קולקציה — כדי שהעיגונים (anchor_ref) ימשיכו לעבוד
QDRANT_URL = "http://localhost:6333"
MEM_TIER = "max"                          # בקלאוד GPU יש RAM בשפע — אין סיבה ל-ssd tier האיטי כאן
BATCH = 512                               # שתי השכבות החדשות קטנות; לא צריך את ה-2048 של הריצה המלאה

SMOKE_TEST = True     # True: מריץ רק על תת-קבוצה קטנה של כל שכבה חדשה כדי לוודא שהצינור עובד
SMOKE_LIMIT = 200

tok = getpass.getpass("HF write token: ")


### 3. הרמת Qdrant מקומי (Docker) — כמו `docker-compose.yml` הקיים

אם `docker` לא זמין בסביבת Lightning שלכם, הריצו Qdrant בדרך אחרת (בינארי מקומי / שירות מנוהל)
ועדכנו `QDRANT_URL` בהתאם — שאר המחברת לא תלויה ב-Docker ספציפית, רק בכתובת שרת Qdrant חי.

In [ ]:
import subprocess, time, requests

subprocess.run(["docker", "run", "-d", "--name", "chavruta-qdrant", "-p", "6333:6333",
                "-v", "qdrant_storage:/qdrant/storage", "qdrant/qdrant"], check=False)

for attempt in range(30):
    try:
        r = requests.get(f"{QDRANT_URL}/collections", timeout=5)
        if r.status_code == 200:
            print("Qdrant is up.")
            break
    except requests.RequestException:
        pass
    time.sleep(2)
else:
    raise RuntimeError("Qdrant did not come up — check `docker logs chavruta-qdrant`")


### 4. שחזור האינדקס הקיים — משתמש בסקריפט הקיים, לא כותב מחדש

`scripts/restore_commercial_snapshot.py` הוא כבר **הדרך הנכונה היחידה** לאכלס את
`chavruta_commercial` (כך כתוב בראש הקובץ עצמו). מריצים אותו כפי שהוא, כ-subprocess, עם משתני
סביבה — לא reimplementing את מנגנון ה-snapshot recovery.

In [ ]:
import os

env = os.environ.copy()
env["INDEX_REPO"] = EXISTING_INDEX_REPO
env["CHAVRUTA_QDRANT_URL"] = QDRANT_URL
env["HF_TOKEN"] = tok
# בלי QDRANT_CONTAINER/QDRANT_SNAPSHOT_DIR: מכונת GPU בענן בד"כ עם RAM בשפע ביחס לגודל ה-snapshot,
# כך שההעלאה הישירה ב-HTTP (הנתיב הפשוט בסקריפט) בטוחה. אם ה-restore נכשל/swap — הוסיפו
# QDRANT_CONTAINER=chavruta-qdrant לסביבה (ראה האזהרה בראש restore_commercial_snapshot.py).

r = subprocess.run(["python", "chavruta_repo/scripts/restore_commercial_snapshot.py"],
                   env=env, capture_output=True, text=True)
print(r.stdout[-3000:])
if r.returncode != 0:
    print("STDERR:", r.stderr[-3000:])
    raise RuntimeError("restore_commercial_snapshot.py failed — לא ממשיכים על קולקציה חלקית")

# ספירת בדיקה: אמור להיות ~2.4M
from qdrant_client import QdrantClient
client = QdrantClient(url=QDRANT_URL, timeout=60)
count_before = client.count(COLLECTION, exact=False).count
print(f"points after restore: {count_before:,} (מצופה: ~2,403,599)")


### 5. הורדה + מיזוג שתי השכבות החדשות בלבד

**לא** `step_merge` המקורי (15 שכבות) — רק שתי הקבצים החדשים.

In [ ]:
from pathlib import Path
from huggingface_hub import hf_hub_download

MERGED = Path("new_tiers_merged.jsonl")
total = 0
with MERGED.open("w", encoding="utf-8") as out:
    for repo in NEW_TIER_REPOS:
        slug = repo.rsplit("-", 1)[-1] if "wikisource-" not in repo else repo.split("wikisource-")[-1]
        fname = f"wikisource_{slug}.jsonl"
        local = hf_hub_download(repo_id=repo, filename=fname, repo_type="dataset", token=tok)
        n = 0
        for line in Path(local).open(encoding="utf-8"):
            if line.strip():
                out.write(line if line.endswith("\n") else line + "\n")
                n += 1
        total += n
        print(f"  + {repo:50s} {n:>7,} chunks")
print(f"[merge] {total:,} new chunks -> {MERGED}")


### 6. הטמעה (bge-m3, GPU) — **אותו קוד בדיוק** כמו `index_commercial_job.py::step_embed`

מועתק במכוון (לא import, כי המקור כתוב כ-script עם משתני מודול גלובליים) — ההתנהגות חייבת
להיות זהה ל-15 השכבות הקיימות, אחרת הווקטורים החדשים לא יהיו באותו "שפה" כמו הקיימים.

In [ ]:
import json as _json
import numpy as np
import torch
from FlagEmbedding import BGEM3FlagModel

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[embed] bge-m3 on {device.upper()} | batch={BATCH}")
if device == "cpu":
    print("[embed] ⚠️  no GPU — this will be very slow")

chunks, bad = [], 0
for line in MERGED.open(encoding="utf-8"):
    line = line.strip().lstrip("\ufeff")
    if not line:
        continue
    try:
        c = _json.loads(line)
    except json.JSONDecodeError:
        bad += 1; continue
    if c.get("document") and c.get("id"):
        chunks.append(c)
    else:
        bad += 1
if bad:
    print(f"[embed] skipped {bad:,} unparseable/incomplete lines")

if SMOKE_TEST:
    chunks = chunks[:SMOKE_LIMIT]
    print(f"[embed] SMOKE_TEST — capping to {len(chunks):,} chunks")

docs = [c["document"] for c in chunks]
print(f"[embed] {len(docs):,} chunks")

model = BGEM3FlagModel("BAAI/bge-m3", use_fp16=(device == "cuda"), device=device)
OUT = Path("out"); OUT.mkdir(exist_ok=True)
dense_parts, sparse_rows = [], []
for s in range(0, len(docs), BATCH):
    enc = model.encode(docs[s:s + BATCH], batch_size=BATCH, max_length=512,
                       return_dense=True, return_sparse=True, return_colbert_vecs=False)
    dense_parts.append(np.asarray(enc["dense_vecs"], dtype="float32"))
    for w in enc["lexical_weights"]:
        sparse_rows.append({int(t): float(v) for t, v in dict(w).items()})
    print(f"  {min(s + BATCH, len(docs)):,}/{len(docs):,}", flush=True)

vecs = np.vstack(dense_parts)
norms = np.linalg.norm(vecs, axis=1, keepdims=True)
norms[norms == 0] = 1.0
vecs /= norms

np.save(str(OUT / "corpus_vectors.npy"), vecs)
with (OUT / "corpus_sparse.jsonl").open("w", encoding="utf-8") as f:
    for i, row in enumerate(sparse_rows):
        f.write(_json.dumps({"i": i, "sparse": row}) + "\n")
with (OUT / "corpus_meta.jsonl").open("w", encoding="utf-8") as f:
    for i, c in enumerate(chunks):
        f.write(_json.dumps({"i": i, "id": c["id"], "document": c["document"],
                            "metadata": c["metadata"]}, ensure_ascii=False) + "\n")
print(f"[embed] ✅ {vecs.shape[0]:,}×{vecs.shape[1]} -> {OUT}")


### 7. `upsert` על גבי הקולקציה הקיימת — **בלי** drop

זה ההבדל היחיד והמכוון מ-`step_load` המקורי: שם מוחקים ובונים מחדש (rebuild מלא); כאן
מוסיפים. `store.upsert` הוא idempotent לפי `chunk_id` — הרצה חוזרת לא משכפלת.

In [ ]:
from chavruta.corpus.ingest import load_processed_chunks
from chavruta.store.qdrant_store import QdrantStore

store = QdrantStore(mode="server", url=QDRANT_URL)
store.ensure_collection(COLLECTION, dim=1024, mem_tier=MEM_TIER)   # no-op: הקולקציה כבר קיימת מהשחזור

batch, total = [], 0
for sc in load_processed_chunks(str(OUT)):
    batch.append(sc)
    if len(batch) >= 250:
        store.upsert(COLLECTION, batch)
        total += len(batch); batch = []
if batch:
    store.upsert(COLLECTION, batch); total += len(batch)
print(f"[upsert] ✅ {total:,} new points added to \'{COLLECTION}\'")

count_after = QdrantClient(url=QDRANT_URL, timeout=60).count(COLLECTION, exact=False).count
print(f"points before: {count_before:,}  after: {count_after:,}  (Δ = {count_after - count_before:,})")
assert count_after >= count_before, "הקולקציה הצטמצמה — regression, לא ממשיכים לפרסום"


### 8. Payload indexes — פעולה בטוחה לחזור עליה, גם אם כבר קיימים

In [ ]:
from qdrant_client.http import models

client = QdrantClient(url=QDRANT_URL, timeout=300)
for field in ("ref", "anchor_ref", "license_he", "license_en"):
    try:
        client.create_payload_index(collection_name=COLLECTION, field_name=field,
                                    field_schema=models.PayloadSchemaType.KEYWORD, wait=True)
        print(f"  ✓ index on \'{field}\'")
    except Exception as exc:
        print(f"  • \'{field}\': {exc}")


### 9. פרסום snapshot **למאגר חדש** — לא דורס את הקיים

זהה ל-`step_publish` המקורי, רק ה-repo היעד שונה (`NEW_INDEX_REPO`).

In [ ]:
from huggingface_hub import HfApi, create_repo

print("creating collection snapshot…")
snap = client.create_snapshot(collection_name=COLLECTION, wait=True)
snap_path = OUT / snap.name
with requests.get(f"{QDRANT_URL}/collections/{COLLECTION}/snapshots/{snap.name}",
                  stream=True, timeout=1800) as r:
    r.raise_for_status()
    with snap_path.open("wb") as f:
        for chunk in r.iter_content(chunk_size=1 << 20):
            f.write(chunk)
print(f"snapshot {snap_path.stat().st_size / 1e9:.2f} GB")

create_repo(NEW_INDEX_REPO, repo_type="dataset", exist_ok=True, token=tok)
api = HfApi()
api.upload_file(path_or_fileobj=str(snap_path), path_in_repo=f"snapshots/{snap.name}",
                repo_id=NEW_INDEX_REPO, repo_type="dataset", token=tok)
manifest = {
    "collection": COLLECTION, "snapshot": f"snapshots/{snap.name}", "mem_tier": MEM_TIER,
    "dim": 1024, "vectors": "dense+sparse (bge-m3)",
    "base_index": EXISTING_INDEX_REPO, "added_tiers": NEW_TIER_REPOS,
    "points_before": count_before, "points_after": count_after,
    "smoke_test": SMOKE_TEST,
}
(OUT / "manifest.json").write_text(_json.dumps(manifest, ensure_ascii=False, indent=2))
api.upload_file(path_or_fileobj=str(OUT / "manifest.json"), path_in_repo="manifest.json",
                repo_id=NEW_INDEX_REPO, repo_type="dataset", token=tok)
print(f"\n✅ https://huggingface.co/datasets/{NEW_INDEX_REPO}")
if SMOKE_TEST:
    print("\n⚠️  זה היה SMOKE_TEST=True — רק חלק מהצ\'אנקים החדשים נכנסו. הגדירו SMOKE_TEST=False להרצה מלאה.")


## 10. לפני שמעבירים פרודקשן לזה

1. **לוודא ש-`anchor_ref` של משנה ברורה אמיתי** — זה בדיוק תא הבדיקה מסעיף 11 במחברת הקודמת,
   עכשיו עם גישה אמיתית לקולקציה: `client.scroll(COLLECTION, scroll_filter=Filter(must=[FieldCondition(key="ref", match=MatchValue(value=<ref שו"ע אמיתי>))]))` צריך להחזיר משהו.
2. **eval retrieval** — להריץ לפני/אחרי ולוודא שאין נסיגה (מדיניות `model-runs-need-approval` —
   eval שקורא למודל צריך אישור; retrieval-only לא).
3. רק אז: לעדכן את `INDEX_REPO` ב-`restore_commercial_snapshot.py` (או `.env`) מ-`chavruta-commercial-index`
   ל-`chavruta-extended-index-v1`, ולהריץ בפרודקשן.
